# Finite Differences

## Midpoint Method

The midpoint method formula is:

$$y(x+h) = y(x) + hf[x+\frac{h}{2}, y(x) + \frac{h}{2}f(x,y(x))]

To check whether this is accurate to the second order, we need to take the Taylor expansion of a given function, say:

$$ y(x+h) = y(x) +h f (x,y(x)) + \frac{h^2}{2}[\frac{\partial f}{\partial x} + f \frac{\partial f}{\partial y}] + O(h^3)$$

And confirm that with the midpoint method, we can achieve this second order approximation.
$$y(x+h) = y(x) +h f (x,y(x)) + \frac{h^2}{2}[\frac{\partial f}{\partial x} + f \frac{\partial f}{\partial y}]$$

Therefore, if we take the approximation from the midpoint method, then the only remaining term would be $O(h^3)$


We can define the step by step of the midpoint method by:
$$k_1 = f(t_n, y_n), \qquad k_2 = f\!\left(t_n + \tfrac{h}{2},\; y_n + \tfrac{h}{2}k_1\right), \qquad y_{n+1} = y_n + h\,k_2.$$


$$k_2 = f\!\left(t_n + \tfrac{h}{2},\; y_n + \tfrac{h}{2}f\right) = f + \frac{h}{2}f_t + \frac{h}{2}f\,f_y + O(h^2),$$

Therefore:
$$y_{n+1} = y_n + h\,k_2 = y_n + hf + \frac{h^2}{2}f_t + \frac{h^2}{2}f\,f_y + O(h^3). \tag{2}$$


When we use the local truncation error, we see these cancel out exactly

##  Euler and fourth-order Runge-Kutta differential equation methods

To solve this dynamics system problem, we can convert this into a system of differential equations to solve:
$$\ddot{x} + x = 0$$

We can rewrite this to be a first order differential of some form so that:
$$\dot{u} = f(x) = \bmatrix{u_2 // -u_1}$$

Solving this first with Euler:

$$u_{n+1} = u_n + h f(t_n, u_n)$$

Solving with Runge-Kutta:
$$k_1 = hf(x,y(x)) = f(t_n, u_n)$$ 
$$k_2 = hf(x+\frac{h}{2}, y(x) + \frac{k_1}{2}) = f(t_n +\frac{h}{2}, u_n + \frac{h}{2}k_1)$$
$$k_3 = hf(x+\frac{h}{2}, y(x)+ \frac{k_2}{2}) = f(t_n + \frac{h}{2}, u_n + \frac{h}{2}k_2)$$
$$k_4 = hf(x+h, y(x)+k_3) = f(t_n+h,u_n+hk_3)$$

Therefore:
$$u_{n+1} = \frac{h}{6}(k_1+2k_2+2k_3+k_4)$$


In [29]:
import numpy as np

def euler_method(f, t0, T, u0, h, *params):
    t, u = t0, np.asarray(u0, dtype=float)
    ts, us = [t], [u.copy()]
    while t < T - 1e-12:
        u = u + h * np.asarray(f(t, u, *params), dtype=float)
        t += h
        ts.append(t); us.append(u.copy())
    return np.array(ts), np.array(us)


def runge_kutta_method(f, t0, T, u0, h, *params):
    t, u = t0, np.asarray(u0, dtype=float)
    ts, us = [t], [u.copy()]
    while t < T - 1e-12:
        k1 = np.asarray(f(t,       u,          *params), dtype=float)
        k2 = np.asarray(f(t + h/2, u + h/2*k1, *params), dtype=float)
        k3 = np.asarray(f(t + h/2, u + h/2*k2, *params), dtype=float)
        k4 = np.asarray(f(t + h,   u + h*k3,   *params), dtype=float)
        u  = u + h/6 * (k1 + 2*k2 + 2*k3 + k4)
        t += h
        ts.append(t); us.append(u.copy())
    return np.array(ts), np.array(us)

In [32]:
def f_harmonic_oscillator(t, u):
    return np.array([u[1], -u[0]])

t_s, us = runge_kutta_method(f_harmonic_oscillator, 0, 2*np.pi, [1.0, 0.0], h=0.01)
t_s, us = euler_method(f_harmonic_oscillator, 0, 2*np.pi, [1.0, 0.0], h=0.01)
x, x_dot = us[:, 0], us[:, 1]


## Pendulum motion

The pendelums motion is described by:
$$l\ddot{\theta} + (g+\ddot{z})sin(\theta) = 0$$

To numerically solve this system, we can again conver this into a system of two first order differential equations for $\theta$ and $z$ where $u = (\theta, \dot{\theta})$ for $\theta$, and for $z$, we can directly rewrite this from $z(t) = Acos(\omega t)$, which becomes:
$$ \ddot{z} = -A\omega^2 cost(\omega t)$$

Therefore the total system becomes:

$$\dot{\mathbf{u}} = \begin{bmatrix} u_2 \\ -\dfrac{g - A\omega^2\cos(\omega t)}{l}\,\sin u_1 \end{bmatrix}$$

We can reuse the components we made for the first problem here 


$$k_1 = \begin{pmatrix} \dot\theta_n \\ -\frac{g - A\Omega^2\cos(\Omega t_n)}{l}\sin\theta_n \end{pmatrix}$$

$$k_2 = \begin{pmatrix} \dot\theta_n + \frac{h}{2}k_1^{(2)} \\ -\frac{g - A\Omega^2\cos(\Omega(t_n+h/2))}{l}\sin\!\left(\theta_n + \frac{h}{2}k_1^{(1)}\right) \end{pmatrix}$$

and similarly for $k_3$, $k_4$, then:

$$\theta_{n+1} = \theta_n + \frac{h}{6}(k_1^{(1)} + 2k_2^{(1)} + 2k_3^{(1)} + k_4^{(1)}),$$
$$\dot\theta_{n+1} = \dot\theta_n + \frac{h}{6}(k_1^{(2)} + 2k_2^{(2)} + 2k_3^{(2)} + k_4^{(2)}).$$

In [ ]:
g = 9.81
l  = 1

def f_pendulum(t, u, A, Omega):
    z_ddot = -A * Omega**2 * np.cos(Omega * t)
    return np.array([u[1], -(g + z_ddot) / l * np.sin(u[0])])

omega0 = np.sqrt(g / l)

ts, us = runge_kutta_method(f_pendulum, 0, 80, [0.05, 0.0], 0.002, 0.1, 2*omega0)

for Omega_ratio in [1.8, 2.0, 2.2]:
    ts, us = runge_kutta_method(f_pendulum, 0, 80, [0.05, 0.0], 0.002,
                 0.1, Omega_ratio * omega0)
    print(f"Omega/omega0 = {Omega_ratio:.1f}, max|theta| = {np.max(np.abs(us[:,0])):.4f}")



Omega/omega0 = 1.8, max|theta| = 0.0500
Omega/omega0 = 2.0, max|theta| = 1.7351
Omega/omega0 = 2.2, max|theta| = 0.7053


In [36]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display
style  = {'description_width': '160px'}
layout = widgets.Layout(width='480px')

w_A     = widgets.FloatSlider(value=0.1,  min=0.0,  max=2.0, step=0.05,
                               description='A', style=style, layout=layout)
w_Omega = widgets.FloatSlider(value=2.0,  min=0.5,  max=4.0, step=0.05,
                               description='Omega/omega0',   style=style, layout=layout)
w_theta0= widgets.FloatSlider(value=0.2,  min=-np.pi, max=np.pi, step=0.05,
                               description='Theta(initial angle, rad)',  style=style, layout=layout)
w_dth0  = widgets.FloatSlider(value=0.0,  min=-5.0, max=5.0,  step=0.1,
                               description='Theta dot(initial velocity)',    style=style, layout=layout)
w_T     = widgets.FloatSlider(value=40.0, min=5.0,  max=200.0, step=5.0,
                               description='T  (integration time, s)',  style=style, layout=layout)
w_method= widgets.ToggleButtons(options=['Runge Kutta', 'Euler'],
                                 description='Method:', style={'description_width': '60px'})

out = widgets.Output()

def run(_=None):
    A     = w_A.value
    Omega = w_Omega.value * omega0
    u0    = [w_theta0.value, w_dth0.value]
    T_end = w_T.value
    h     = 0.005
    solve = runge_kutta_method if w_method.value == 'Runge Kutta' else euler_method

    ts, us = solve(f_pendulum, 0, T_end, u0, h, A, Omega)
    theta  = us[:, 0]
    dtheta = us[:, 1]
    theta_wrap = (theta + np.pi) % (2 * np.pi) - np.pi

    color_t = ts / T_end

    fig = make_subplots(
        rows=2, cols=1,
        subplot_titles=('Theta vs. Theta dot',
                        'Theta dot vs. time'),
        vertical_spacing=0.18
    )

    # ── Panel 1: phase portrait ────────────────────────────────────────────────
    fig.add_trace(go.Scatter(x=theta_wrap, y=dtheta,
                             mode='markers',
                             marker=dict(size=2, color=color_t,
                                         colorscale='Plasma',
                                         colorbar=dict(title='t/T', len=0.45,
                                                        y=0.78, x=1.02)),
                             name='phase'), row=1, col=1)

    # ── Panel 2: dtheta(t) ───────────────────────────────────────────────────
    fig.add_trace(go.Scatter(x=ts, y=dtheta, mode='lines',
                             line=dict(color='blue', width=1.2),
                             name='Theta dot(t)'), row=2, col=1)

    fig.update_xaxes(title_text='Theta(rad)', row=1, col=1)
    fig.update_xaxes(title_text='Time(s)', row=2, col=1)
    fig.update_yaxes(title_text='Theta dot(rad/s)', row=1, col=1)
    fig.update_yaxes(title_text='Theta dot(rad/s)', row=2, col=1)

    title = (f'A = {A:.2f} m,  Omega/omega0 = {w_Omega.value:.2f},  '
             f'Theta(initial angle, rad) = {w_theta0.value:.2f},  method = {w_method.value}')
    fig.update_layout(height=620, showlegend=False,
                      title=dict(text=title, font=dict(size=13)),
                      margin=dict(t=80, b=40, l=60, r=80))

    with out:
        out.clear_output(wait=True)
        fig.show()

btn = widgets.Button(description='Run', button_style='primary',
                     layout=widgets.Layout(width='120px', height='36px'))
btn.on_click(run)

left  = widgets.VBox([w_A, w_Omega, w_theta0])
right = widgets.VBox([w_dth0, w_T, w_method])
controls = widgets.VBox([widgets.HBox([left, right]), btn])

display(controls, out)
run()


Output()

In [ ]:
w_A_anim = widgets.Dropdown(options=[(f'{x:.2f}', x) for x in [0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.4, 0.5, 0.75, 1.0, 1.5]],
                             value=0.1, description='A (m)', style=style, layout=widgets.Layout(width='180px'))
out_anim = widgets.Output()

def run_anim(_=None):
    A     = w_A_anim.value if w_A_anim.value is not None else 0.1
    Omega = 2.0 * omega0
    T_end = 15.0
    h     = 0.002

    ts, us = runge_kutta_method(f_pendulum, 0, T_end, [0.2, 0.0], h, A, Omega)
    theta  = us[:, 0]
    stride = max(1, len(ts) // 600)
    ts_d, theta_d = ts[::stride], theta[::stride]

    pivot_y = A * np.cos(Omega * ts_d)      # pivot moves vertically
    bob_x   = l * np.sin(theta_d)
    bob_y   = pivot_y - l * np.cos(theta_d)

    pad     = l * 1.4 + A
    frames = []
    for k in range(len(ts_d)):
        frames.append(go.Frame(
            data=[
                go.Scatter(x=[0, bob_x[k]], y=[pivot_y[k], bob_y[k]],
                           mode='lines', line=dict(color='#31688e', width=3)),
                go.Scatter(x=[0], y=[pivot_y[k]], mode='markers',
                           marker=dict(size=10, color='#21918c')),
                go.Scatter(x=[bob_x[k]], y=[bob_y[k]], mode='markers',
                           marker=dict(size=18, color='#fde725')),
                go.Scatter(x=bob_x[:k+1], y=bob_y[:k+1],
                           mode='lines',
                           line=dict(color='rgba(53,183,121,0.4)', width=1.5))
            ],
            name=str(k),
            layout=go.Layout(title_text=f't = {ts_d[k]:.2f} s  |  θ = {np.degrees(theta_d[k]):.1f}°')
        ))

    fig = go.Figure(
        data=frames[0].data,
        layout=go.Layout(
            title=f'A={A:.2f} m',
            xaxis=dict(range=[-pad, pad], zeroline=True, title='x (m)',
                       scaleanchor='y', scaleratio=1),
            yaxis=dict(range=[-pad, pad], zeroline=True, title='y (m)'),
            height=520, width=540,
            showlegend=False,
            updatemenus=[dict(
                type='buttons', showactive=False,
                buttons=[
                    dict(label='Play',
                         method='animate',
                         args=[None, dict(frame=dict(duration=30, redraw=True),
                                          fromcurrent=True)]),
                    dict(label='Pause',
                         method='animate',
                         args=[[None], dict(frame=dict(duration=0, redraw=False),
                                             mode='immediate')])
                ]
            )],
            sliders=[dict(
                steps=[dict(args=[[f.name],
                                   dict(mode='immediate',
                                        frame=dict(duration=0, redraw=True))],
                            method='animate', label='')
                       for f in frames],
                transition=dict(duration=0),
                x=0.05, y=0, len=0.9,
                currentvalue=dict(visible=False)
            )]
        ),
        frames=frames
    )

    with out_anim:
        out_anim.clear_output(wait=True)
        fig.show()

btn_anim = widgets.Button(description='Animate',
                           layout=widgets.Layout(width='130px', height='36px'))
btn_anim.on_click(run_anim)

display(widgets.HBox([
    widgets.VBox([w_A_anim, btn_anim],
                 layout=widgets.Layout(width='200px', margin='0 20px 0 0')),
    out_anim
], layout=widgets.Layout(align_items='flex-start', width='100%')))

In [ ]:
import ipywidgets as widgets

slider_layout = widgets.Layout(width='300px')
def f_spherical_pendulum(t, u, A, Omega):
    theta, phi, th_dot, ph_dot = u[0], u[1], u[2], u[3]
    z_ddot = -A * Omega**2 * np.cos(Omega * t)
    sin_th, cos_th = np.sin(theta), np.cos(theta)
    th_ddot = sin_th * cos_th * ph_dot**2 - (g + z_ddot) / l * sin_th
    ph_ddot = -2 * th_dot * ph_dot * cos_th / (sin_th + 1e-12) if abs(sin_th) > 1e-8 else 0
    return np.array([th_dot, ph_dot, th_ddot, ph_ddot])

w_A_3d = widgets.FloatSlider(value=0.1, min=0.0, max=1.5, step=0.05,
description='A (m)', style=style, layout=widgets.Layout(width='380px'))
w_Or_3d = widgets.FloatSlider(value=2.0, min=0.5, max=4.0, step=0.05,
description='Ω/ω₀', style=style, layout=widgets.Layout(width='380px'))
w_T_3d = widgets.FloatSlider(value=15.0, min=3.0, max=40.0, step=1.0,
description='T (s)', style=style, layout=widgets.Layout(width='380px'))
w_phi0 = widgets.FloatSlider(value=0.3, min=0.0, max=np.pi, step=0.05,
description='φ₀ (rad)', style=style, layout=widgets.Layout(width='380px'))
out_3d = widgets.Output()

w_A_3d.layout  = slider_layout
w_Or_3d.layout = slider_layout
w_T_3d.layout  = slider_layout
w_phi0.layout  = slider_layout
def run_anim_3d(_=None):
    A     = w_A_3d.value
    Omega = w_Or_3d.value * omega0
    T_end = w_T_3d.value
    phi0  = w_phi0.value
    h     = 0.008

    u0 = [0.2, phi0, 0.0, 0.0]
    ts, us = runge_kutta_method(f_spherical_pendulum, 0, T_end, u0, h, A, Omega)
    
    stride = max(1, len(ts) // 400)
    ts_d = ts[::stride]
    theta_d = us[::stride, 0]
    phi_d = us[::stride, 1]

    pivot_z = A * np.cos(Omega * ts_d)
    bob_x = l * np.sin(theta_d) * np.cos(phi_d)
    bob_y = l * np.sin(theta_d) * np.sin(phi_d)
    bob_z = pivot_z - l * np.cos(theta_d)

    limit = l + A + 0.1 

    frames = []
    for k in range(len(ts_d)):
        pivot_trace = go.Scatter3d(
            x=[0], y=[0], z=[pivot_z[k]],
            mode='markers', marker=dict(size=6, color='#440154') # Viridis Purple
        )
        rod_trace = go.Scatter3d(
            x=[0, bob_x[k]], y=[0, bob_y[k]], z=[pivot_z[k], bob_z[k]],
            mode='lines', line=dict(color='#3b528b', width=5) # Viridis Blue
        )
        bob_trace = go.Scatter3d(
            x=[bob_x[k]], y=[bob_y[k]], z=[bob_z[k]],
            mode='markers', marker=dict(size=10, color='#fde725') # Viridis Yellow
        )
        trail_trace = go.Scatter3d(
            x=bob_x[:k+1], y=bob_y[:k+1], z=bob_z[:k+1],
            mode='lines', line=dict(color='rgba(33,145,140,0.5)', width=3)
        )
        
        frames.append(go.Frame(data=[pivot_trace, rod_trace, bob_trace, trail_trace], name=str(k)))

    fig = go.Figure(
        data=frames[0].data,
        layout=go.Layout(
            title=f'3D Pendulum Simulation',
            paper_bgcolor='white',
            plot_bgcolor='white',
            scene=dict(
                xaxis=dict(range=[-limit, limit], showgrid=False, zeroline=False, showbackground=False, title='', showticklabels=False),
                yaxis=dict(range=[-limit, limit], showgrid=False, zeroline=False, showbackground=False, title='', showticklabels=False),
                zaxis=dict(range=[-limit, limit], showgrid=False, zeroline=False, showbackground=False, title='', showticklabels=False),
                aspectmode='cube'
            ),
            uirevision='constant', 
            margin=dict(l=0, r=0, b=0, t=40),
            height=600, width=600,
            showlegend=False,
            updatemenus=[dict(
                type='buttons', showactive=False,
                y=0, x=0.1,
                buttons=[
                    dict(label='▶ Play', method='animate',
                         args=[None, dict(frame=dict(duration=20, redraw=True), fromcurrent=True)]),
                    dict(label='|| Pause', method='animate',
                         args=[[None], dict(frame=dict(duration=0, redraw=False), mode='immediate')])
                ]
            )]
        ),
        frames=frames
    )

    with out_3d:
        out_3d.clear_output(wait=True)
        fig.show()

btn_3d = widgets.Button(description='Animate 3D',
                        layout=widgets.Layout(width='130px', height='36px'))
btn_3d.on_click(run_anim_3d)

controls_vbox = widgets.VBox([
    widgets.HTML("<b>Simulation Parameters</b>"),
    w_A_3d, w_Or_3d, w_T_3d, w_phi0, 
    btn_3d
], layout=widgets.Layout(min_width='320px', padding='20px'))

final_ui = widgets.HBox([controls_vbox, out_3d], layout=widgets.Layout(align_items='center'))

display(final_ui)

## Simulating strings with wave equation

The wave equation is given to us by:
$$\frac{\partial^2u}{\partial t^2} = v^2 \frac{\partial^2 u}{\partial x^2} + \gamma \frac{\partial}{\partial t} \frac{\partial^2 u}{\partial t^2}$$

We can assume there are fixed boundary conditions of the string being fixed on both ends (e.g., a violin string), which means that the string is set at $u(0,t) = u(1,t) = 0$

We can say that the "pluck" is going to happen at position x, and we can solve this with separateion of variables:
$u(x,t) = X(x)\,T(t)$ into $u_{tt} = v^2 u_{xx}$:

$$\frac{\ddot{T}}{v^2 T} = \frac{X''}{X} = -k^2$$

Using the boundary constiions that we defined above $X(0) = X(L) = 0$ result in:

$$k_n = \frac{n\pi}{L}, \qquad X_n(x) = \sin(k_n x), \qquad n = 1, 2, 3, \ldots$$

Therefore fo rthe time portion of the solution we get that $\ddot{T}_n + \omega_n^2 T_n = 0$:

$$\omega_n = v k_n = \frac{n\pi v}{L}$$

The general solution therefore becomes:

$$u(x,t) = \sum_{n=1}^{\infty} A_n \sin\!\left(\frac{n\pi x}{L}\right)\cos(\omega_n t)$$

$$A_n = \frac{2}{L}\int_0^L f(x)\sin\!\left(\frac{n\pi x}{L}\right)dx$$



This above is true for an undamped case, but then we also need to determine what happens when we add damping

With damping, we update the substition to be $u = X(x)\,T(t)$ and use $X_n = \sin(k_n x)$:

$$\ddot{T}_n + \gamma k_n^2 \dot{T}_n + \omega_n^2 T_n = 0$$

Therefore,

$$\zeta_n = \frac{\gamma k_n^2}{2\omega_n} = \frac{\gamma n\pi}{2vL}$$

For $\zeta_n < 1$, the damped frequency is:

$$\omega_n^d = \omega_n\sqrt{1 - \zeta_n^2}$$


With $\dot{u}(x,0) = 0$, we get $B_n = A_n \zeta_n / \sqrt{1 - \zeta_n^2}$.

Then the full damped sol is:

$${u(x,t) = \sum_{n=1}^{\infty} A_n \frac{e^{-\zeta_n\omega_n t}}{\sqrt{1-\zeta_n^2}} \sin(k_n x)\cos\!\left(\omega_n^d t - \phi_n\right)}$$



Numerically, we need to use finite discretization as the approach to solve this (though I think you could also use von Neumann and just directly do the stability analysis)


Index the solution as $u(j\Delta x,\, n\Delta t) = u_j^n$, where $j = 0,\ldots,N$ labels the spatial grid and $n = 0, 1, 2,\ldots$ labels the time level.

Replace each derivative with second-order centred differences:

$$\frac{\partial^2 u}{\partial t^2}\bigg|_j^n \approx \frac{u_j^{n+1} - 2u_j^n + u_j^{n-1}}{\Delta t^2}$$

$$\frac{\partial^2 u}{\partial x^2}\bigg|_j^n \approx \frac{u_{j+1}^n - 2u_j^n + u_{j-1}^n}{\Delta x^2}$$


Subbing all this in let's us get to:
$${u_j^{n+1} = 2u_j^n - u_j^{n-1} + (\alpha + \beta)\!\left(u_{j+1}^n - 2u_j^n + u_{j-1}^n\right) - \beta\!\left(u_{j+1}^{n-1} - 2u_j^{n-1} + u_{j-1}^{n-1}\right)}$$


Therfore, we can get the next update step, and still be first order accurate:
$u_j^1 = u_j^0 + \frac{\alpha + \beta}{2}\!\left(u_{j+1}^0 - 2u_j^0 + u_{j-1}^0\right)$

For the stability, we can linearize and look at the growth of the ansatz (setting $\gamma = 0$, i.e. $\beta = 0$):

$$u_j^n = A(k)^n e^{ijk\Delta x}$$

This simplifies down to:
$$A^2 - 2\!\left(1 - 2\alpha\sin^2\frac{k\Delta x}{2}\right)\!A + 1 = 0$$

**source: used Wolfram Alpha to simplify this


$$A = (1-\mu) \pm \sqrt{(1-\mu)^2 - 1}$$

Require that the magnitude satisfy $|A| \leq 1$ for all wavenumbers $k$:

- If $0 \leq \mu \leq 2$: the discriminant is non-positive, both roots are complex conjugates, and $|A|^2 = (1-\mu)^2 + \bigl(1-(1-\mu)^2\bigr) = 1$. ✓
- If $\mu > 2$: both roots are real and the larger exceeds 1 — any initial condition will diverge. ✗


In [59]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.io import wavfile
from scipy.signal import resample
import ipywidgets as widgets
from IPython.display import display, HTML
import base64, io

def make_L(N):
    return (np.diag(-2 * np.ones(N))
            + np.diag(np.ones(N-1),  1)
            + np.diag(np.ones(N-1), -1))

def pluck(x, L_str, x_peak_frac, height=1.0):
    xn = x / L_str
    xp = x_peak_frac
    return np.where(xn <= xp, height * xn / xp,
                               height * (1 - xn) / (1 - xp))

def solve_string(v=42.0, L_str=0.65, gamma=0.0, Nx=80,
                 T=2.0, CFL=0.9, x_peak=0.3, height=1.0,
                 x_listen=0.8, n_vis_frames=500, sr=22050):
    dx   = L_str / (Nx + 1)
    dt   = CFL * dx / v
    Nt   = int(T / dt)
    r    = v * dt / dx                   
    lamba    = gamma * dt / (2 * dx**2)     

    x    = np.linspace(dx, L_str - dx, Nx)
    j_l  = int(x_listen * Nx)

    Lm   = make_L(Nx)
    A    = np.eye(Nx) - lamba * Lm         

    u0      = pluck(x, L_str, x_peak, height)
    u_prev  = u0.copy()
    u_curr  = np.linalg.solve(A, u0 + 0.5 * r**2 * (Lm @ u0))

    stride    = max(1, Nt // n_vis_frames)
    ts_out    = [0.0];  us_out = [u0.copy()]
    audio_raw = np.zeros(Nt + 1)
    audio_raw[0] = u_prev[j_l]

    def _energy(um, up):
        dudt = (up - um) / (2 * dt)
        dudx = np.gradient(np.r_[0, um, 0], dx)
        return 0.5 * dx * (np.sum(dudt**2) + v**2 * np.sum(dudx**2))

    en_out = [_energy(u_prev, u_curr)]

    for n in range(1, Nt + 1):
        rhs    = (2 * u_curr - u_prev
                  + r**2 * (Lm @ u_curr)
                  + lamba   * (Lm @ u_prev))
        u_next = np.linalg.solve(A, rhs)
        audio_raw[n] = u_curr[j_l]
        u_prev, u_curr = u_curr, u_next
        if n % stride == 0:
            ts_out.append(n * dt)
            us_out.append(u_curr.copy())
            en_out.append(_energy(u_prev, u_curr))
    n_out  = int(T * sr)
    audio  = resample(audio_raw, n_out)
    audio /= np.max(np.abs(audio)) + 1e-12
    audio *= np.linspace(1, 0, n_out) ** 0.4

    return (np.array(ts_out), np.array(us_out),
            x, np.array(en_out), audio, sr)


def audio_widget(signal, sr, label=''):
    """Embed signal as an HTML <audio> element (WAV data URI)."""
    pcm = (np.clip(signal, -1, 1) * 32767).astype(np.int16)
    buf = io.BytesIO()
    wavfile.write(buf, sr, pcm)
    b64 = base64.b64encode(buf.getvalue()).decode()
    return HTML(
        f'<div style="margin:6px 0"><b>{label}</b><br>'
        f'<audio controls style="width:420px">'
        f'<source src="data:audio/wav;base64,{b64}" type="audio/wav">'
        f'</audio></div>'
    )


def harmonic_amplitudes(ts, us, N_modes=6):
    Nx   = us.shape[1]
    xn   = np.linspace(1/(Nx+1), Nx/(Nx+1), Nx)
    amps = np.zeros((len(ts), N_modes))
    for n in range(1, N_modes + 1):
        mode = np.sin(n * np.pi * xn)
        amps[:, n-1] = np.abs(us @ mode) / (Nx / 2)
    return amps


In [68]:
cases = [
    dict(gamma=0.0,   label='\u03b3 = 0', color = "blue"),
    dict(gamma=0.005, label='\u03b3 = 0.005', color = "green"),
    dict(gamma=0.030, label='\u03b3 = 0.030', color = "red")
]
results = {}
for c in cases:
    ts, us, x, en, audio, sr = solve_string(gamma=c['gamma'], T=2.0, n_vis_frames=400)
    results[c['label']] = dict(ts=ts, us=us, x=x, en=en, audio=audio, sr=sr, **c)

fig = go.Figure()
for lbl, d in results.items():
    fig.add_trace(go.Scatter(x=d['ts'], y=d['en']/d['en'][0],
                             name=lbl, mode='lines',
                             line=dict(color=d['color'], width=2)))

fig.update_xaxes(title_text='t (s)')
fig.update_yaxes(title_text='Energy / Initial Energy')
fig.update_layout(height=380, title='DAmping impacts', margin=dict(t=60))
fig.show()

/var/folders/37/mbm5qm1d03lg63qfy97k4tmw0000gn/T/ipykernel_48053/1314154642.py:38: RuntimeWarning:

divide by zero encountered in matmul

/var/folders/37/mbm5qm1d03lg63qfy97k4tmw0000gn/T/ipykernel_48053/1314154642.py:38: RuntimeWarning:

overflow encountered in matmul

/var/folders/37/mbm5qm1d03lg63qfy97k4tmw0000gn/T/ipykernel_48053/1314154642.py:38: RuntimeWarning:

invalid value encountered in matmul

/var/folders/37/mbm5qm1d03lg63qfy97k4tmw0000gn/T/ipykernel_48053/1314154642.py:54: RuntimeWarning:

divide by zero encountered in matmul

/var/folders/37/mbm5qm1d03lg63qfy97k4tmw0000gn/T/ipykernel_48053/1314154642.py:54: RuntimeWarning:

overflow encountered in matmul

/var/folders/37/mbm5qm1d03lg63qfy97k4tmw0000gn/T/ipykernel_48053/1314154642.py:54: RuntimeWarning:

invalid value encountered in matmul

/var/folders/37/mbm5qm1d03lg63qfy97k4tmw0000gn/T/ipykernel_48053/1314154642.py:55: RuntimeWarning:

divide by zero encountered in matmul

/var/folders/37/mbm5qm1d03lg63qfy97k4tmw0000gn/T

In [75]:
# Audio players
for lbl, d in results.items():
    display(audio_widget(d['audio'], d['sr'], label=lbl))


What happens if we add material properties to the string?

What happens if we have multiple "plucks" happening within short sequence of eachother?

In [72]:
sw2 = {'description_width': '100px'}
lw2 = widgets.Layout(width='360px')

w_ag  = widgets.FloatSlider(value=0.004, min=0.0, max=0.04, step=0.002,
                             description='\u03b3', readout_format='.3f',
                             style=sw2, layout=lw2)
w_axp = widgets.FloatSlider(value=0.3,   min=0.05, max=0.95, step=0.05,
                             description='x_peak', style=sw2, layout=lw2)
out_anim = widgets.Output()

def run_anim(_=None):
    ts, us, x, _, audio, sr = solve_string(
        gamma=w_ag.value, T=2.0, CFL=0.9,
        x_peak=w_axp.value, n_vis_frames=250)

    xf = np.r_[0, x, x[-1] + x[1] - x[0]]
    frames = []
    for k in range(len(ts)):
        uf = np.r_[0, us[k], 0]
        frames.append(go.Frame(
            data=[go.Scatter(x=xf, y=uf, mode='lines',
                             line=dict(color='royalblue', width=2.5))],
            name=str(k),
            layout=go.Layout(title_text=f't = {ts[k]:.4f} s')))

    fig = go.Figure(
        data=[go.Scatter(x=xf, y=np.r_[0, us[0], 0], mode='lines',
                         line=dict(color='royalblue', width=2.5))],
        layout=go.Layout(
            title=f'\u03b3={w_ag.value:.3f}, x_peak={w_axp.value:.2f}',
            xaxis=dict(range=[0, xf[-1]*1.02], title='x (m)'),
            yaxis=dict(range=[-1.3, 1.3], title='u(x,t)'),
            height=400, width=700, showlegend=False,
            updatemenus=[dict(
                type='buttons', showactive=False,
                buttons=[
                    dict(label='\u25b6 Play', method='animate',
                         args=[None, dict(frame=dict(duration=20, redraw=True),
                                          fromcurrent=True)]),
                    dict(label='\u23f8 Pause', method='animate',
                         args=[[None], dict(frame=dict(duration=0, redraw=False),
                                             mode='immediate')])
                ])],
            sliders=[dict(
                steps=[dict(args=[[f.name],
                                   dict(mode='immediate',
                                        frame=dict(duration=0, redraw=True))],
                            method='animate', label='') for f in frames],
                x=0.05, y=0, len=0.9,
                currentvalue=dict(visible=False),
                transition=dict(duration=0))])
    , frames=frames)

    with out_anim:
        out_anim.clear_output(wait=True)
        fig.show()
        display(audio_widget(audio, sr,
                             label=f'\u03b3={w_ag.value:.3f}, x_peak={w_axp.value:.2f}'))

btn_anim = widgets.Button(description='Animate & Play', button_style='success',
                           layout=widgets.Layout(width='160px', height='36px'))
btn_anim.on_click(run_anim)
display(widgets.VBox([widgets.HBox([w_ag, w_axp]), btn_anim]), out_anim)
run_anim()

Output()

/var/folders/37/mbm5qm1d03lg63qfy97k4tmw0000gn/T/ipykernel_48053/1314154642.py:38: RuntimeWarning:

divide by zero encountered in matmul

/var/folders/37/mbm5qm1d03lg63qfy97k4tmw0000gn/T/ipykernel_48053/1314154642.py:38: RuntimeWarning:

overflow encountered in matmul

/var/folders/37/mbm5qm1d03lg63qfy97k4tmw0000gn/T/ipykernel_48053/1314154642.py:38: RuntimeWarning:

invalid value encountered in matmul

/var/folders/37/mbm5qm1d03lg63qfy97k4tmw0000gn/T/ipykernel_48053/1314154642.py:54: RuntimeWarning:

divide by zero encountered in matmul

/var/folders/37/mbm5qm1d03lg63qfy97k4tmw0000gn/T/ipykernel_48053/1314154642.py:54: RuntimeWarning:

overflow encountered in matmul

/var/folders/37/mbm5qm1d03lg63qfy97k4tmw0000gn/T/ipykernel_48053/1314154642.py:54: RuntimeWarning:

invalid value encountered in matmul

/var/folders/37/mbm5qm1d03lg63qfy97k4tmw0000gn/T/ipykernel_48053/1314154642.py:55: RuntimeWarning:

divide by zero encountered in matmul

/var/folders/37/mbm5qm1d03lg63qfy97k4tmw0000gn/T